# Lab 1: NumPy for the Original Cats vs Dogs Dataset

In this notebook, you will treat **original Kaggle Dogs vs Cats competition images** as NumPy arrays and build a tiny nearest-centroid baseline.

This version focuses on core NumPy image operations and keeps the workflow concrete:

- load an image into a NumPy array
- crop and flip with slicing
- normalize to `[0, 1]`
- convert RGB to grayscale
- compute summaries with `axis=`
- apply a small filter with a kernel and matrix multiplication
- flatten an image into one vector
- engineer features with `np.concatenate(...)` and `np.apply_along_axis(...)`
- classify cat vs dog with a tiny nearest-centroid baseline

**Dataset assumption**

Use the original Dogs vs Cats competition data extracted into:

`data/dogs_vs_cats_original/train/`

If it is missing, run:

`uv run python scripts/download_original_dogs_vs_cats.py --force`


In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from lab_utils.visualization import (
    plot_centroid_heatmap,
    plot_feature_vector,
    plot_prediction_gallery,
    show_image_gallery,
)


def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "data" / "dogs_vs_cats_original").exists():
            return candidate
    return Path.cwd().resolve()


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "dogs_vs_cats_original"
TRAIN_DIR = DATA_ROOT / "train"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

LABELS = ("cat", "dog")
LABEL_TO_INDEX = {"cat": 0, "dog": 1}
STUDENT_ID = 10422021  # Replace with your own student ID.
SEED = int(STUDENT_ID)
NUMPY_PRED_PATH = ARTIFACT_DIR / f"lab1_numpy_original_predictions_{STUDENT_ID}.csv"
EDGE_KERNEL = np.array(
    [
        [-1.0, 0.0, 1.0],
        [-2.0, 0.0, 2.0],
        [-1.0, 0.0, 1.0],
    ],
    dtype=np.float32,
)


def label_from_path(path: Path) -> str:
    label = path.stem.split(".")[0]
    if label not in LABEL_TO_INDEX:
        raise ValueError(f"Unexpected label in filename: {path.name}")
    return label


def load_preview_image(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"))


def shuffled_paths(paths: list[Path], seed_offset: int = 0) -> list[Path]:
    rng = np.random.default_rng(SEED + seed_offset)
    indices = rng.permutation(len(paths))
    return [paths[int(idx)] for idx in indices]


def split_paths_per_class(paths: list[Path], train_count: int, test_count: int, seed_offset: int) -> tuple[list[Path], list[Path]]:
    ordered = shuffled_paths(paths, seed_offset=seed_offset)
    train_subset = ordered[:train_count]
    test_subset = ordered[train_count:train_count + test_count]
    return train_subset, test_subset


def sample_per_class(paths: list[Path], n_per_class: int, seed_offset: int = 0) -> list[Path]:
    by_label = {label: [path for path in paths if label_from_path(path) == label] for label in LABELS}
    sampled = []
    for label_index, label in enumerate(LABELS):
        ordered = shuffled_paths(by_label[label], seed_offset=seed_offset + 50 * label_index)
        sampled.extend(ordered[:n_per_class])
    return sampled


if not TRAIN_DIR.exists():
    raise FileNotFoundError(
        "Original Dogs vs Cats dataset not found. Run scripts/download_original_dogs_vs_cats.py --force first."
    )

all_cat_paths = sorted(TRAIN_DIR.glob("cat.*.jpg")) + sorted(TRAIN_DIR.glob("cat.*.png"))
all_dog_paths = sorted(TRAIN_DIR.glob("dog.*.jpg")) + sorted(TRAIN_DIR.glob("dog.*.png"))
if not all_cat_paths or not all_dog_paths:
    raise FileNotFoundError(
        "Expected cat.* and dog.* images inside data/dogs_vs_cats_original/train/."
    )

TRAIN_PER_CLASS = 40
TEST_PER_CLASS = 10
cat_train_paths, cat_test_paths = split_paths_per_class(all_cat_paths, TRAIN_PER_CLASS, TEST_PER_CLASS, seed_offset=100)
dog_train_paths, dog_test_paths = split_paths_per_class(all_dog_paths, TRAIN_PER_CLASS, TEST_PER_CLASS, seed_offset=200)
train_paths = shuffled_paths(cat_train_paths + dog_train_paths, seed_offset=300)
test_paths = shuffled_paths(cat_test_paths + dog_test_paths, seed_offset=400)
sample_path = train_paths[0]

print(f"Student ID seed: {STUDENT_ID}")
print(f"Using original competition images from: {TRAIN_DIR}")
print(f"Train subset: {len(train_paths)} images ({TRAIN_PER_CLASS} per class)")
print(f"Test subset: {len(test_paths)} images ({TEST_PER_CLASS} per class)")
print(f"Predictions will be saved to: {NUMPY_PRED_PATH.name}")


### Visual Helper: Preview the Original Dataset

Before starting the TODOs, look at a few original cat and dog images from the student-specific subset.


In [ ]:
preview_paths = sample_per_class(train_paths, n_per_class=3, seed_offset=10)
preview_images = [load_preview_image(path) for path in preview_paths]
preview_titles = [f"{label_from_path(path)}: {path.name}" for path in preview_paths]
show_image_gallery(
    preview_images,
    titles=preview_titles,
    ncols=3,
    figsize=(10, 6),
    suptitle="Original Dogs vs Cats preview",
)
plt.show()


## Question 1: Load one image into a NumPy array

Write a function that:

- opens one file from disk
- converts it to RGB
- returns an `H x W x C` NumPy array

This is the starting point for every later NumPy operation in the lab.


In [ ]:
def load_image_np(path: Path) -> np.ndarray:
    # TODO: open the file, convert it to RGB, and return an H x W x C NumPy array.
    raise NotImplementedError("Load one RGB image into a NumPy array.")


sample_image = load_image_np(sample_path)
print("shape:", sample_image.shape)
print("dtype:", sample_image.dtype)
print("min/max:", sample_image.min(), sample_image.max())
show_image_gallery([sample_image], titles=[sample_path.name], ncols=1, figsize=(4, 4))
plt.show()


## Question 2: Crop the image with slicing

Implement a centered square crop. Keep the crop size at `64 x 64` for the rest of the lab so later operations stay consistent.


In [ ]:
def center_crop(image: np.ndarray, crop_size: int = 64) -> np.ndarray:
    # TODO: compute the center crop indices and return a square crop.
    raise NotImplementedError("Implement a centered crop with slicing.")


cropped_image = center_crop(sample_image, crop_size=64)
print("cropped shape:", cropped_image.shape)
show_image_gallery(
    [sample_image, cropped_image],
    titles=["Original", "Center crop"],
    ncols=2,
    figsize=(8, 4),
)
plt.show()


## Question 3: Flip the crop horizontally

Mirror the cropped image from left to right using slicing only.


In [ ]:
def flip_horizontal(image: np.ndarray) -> np.ndarray:
    # TODO: return a left-right flipped copy using slicing.
    raise NotImplementedError("Flip the image horizontally with slicing.")


flipped_image = flip_horizontal(cropped_image)
show_image_gallery(
    [cropped_image, flipped_image],
    titles=["Cropped", "Flipped"],
    ncols=2,
    figsize=(8, 4),
)
plt.show()


## Question 4: Normalize pixels to `[0, 1]`

Convert the cropped RGB image from unsigned integers into `float32` values in the range `[0, 1]`.


In [ ]:
def normalize_01(image: np.ndarray) -> np.ndarray:
    # TODO: convert the image to float32 and divide by 255.
    raise NotImplementedError("Normalize pixel values to [0, 1].")


sample_float = normalize_01(cropped_image)
print("dtype:", sample_float.dtype)
print("min/max:", sample_float.min(), sample_float.max())
show_image_gallery(
    [cropped_image, sample_float],
    titles=["Cropped uint8", "Normalized float"],
    ncols=2,
    figsize=(8, 4),
)
plt.show()


## Question 5: Convert RGB to grayscale

Turn the normalized RGB image into a single grayscale array using standard RGB weights.


In [ ]:
def rgb_to_gray(image_float: np.ndarray) -> np.ndarray:
    # TODO: convert normalized RGB values to one grayscale image.
    raise NotImplementedError("Convert RGB to grayscale.")


sample_gray = rgb_to_gray(sample_float)
print("gray shape:", sample_gray.shape)
print("gray dtype:", sample_gray.dtype)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample_float)
axes[0].set_title("Normalized RGB")
axes[0].axis("off")
axes[1].imshow(sample_gray, cmap="gray")
axes[1].set_title("Grayscale")
axes[1].axis("off")
fig.tight_layout()
plt.show()


## Question 6: Use `axis=` to summarize channels

Compute one mean value per color channel with `axis=(0, 1)`, then choose the brightest channel with `np.argmax(...)`.


In [ ]:
CHANNEL_NAMES = np.array(["red", "green", "blue"])


def channel_summary(image_float: np.ndarray) -> tuple[np.ndarray, int]:
    # TODO: compute per-channel means with axis=(0, 1) and return the brightest channel index.
    raise NotImplementedError("Summarize the RGB channels with axis=(0, 1).")


sample_channel_means, sample_brightest = channel_summary(sample_float)
print("channel means:", sample_channel_means)
print("brightest channel:", CHANNEL_NAMES[sample_brightest])
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(CHANNEL_NAMES, sample_channel_means, color=["#E74C3C", "#2ECC71", "#3498DB"])
ax.set_title("Average brightness per channel")
ax.set_ylabel("Mean value")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()


## Question 7: Apply a filter with a kernel and matrix multiplication

Implement a tiny 2D convolution on the grayscale image. At each location:

1. take a `3 x 3` patch
2. flatten the patch and kernel
3. multiply them with `@`

Use the Sobel-style kernel from the setup cell.


In [ ]:
def convolve2d_matmul(image_gray: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    # TODO: slide the kernel over the image and use @ on flattened patches and the flattened kernel.
    raise NotImplementedError("Apply a 2D filter with matrix multiplication.")


sample_filtered = convolve2d_matmul(sample_gray, EDGE_KERNEL)
print("filtered shape:", sample_filtered.shape)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample_gray, cmap="gray")
axes[0].set_title("Grayscale")
axes[0].axis("off")
axes[1].imshow(np.abs(sample_filtered), cmap="magma")
axes[1].set_title("Filtered |response|")
axes[1].axis("off")
fig.tight_layout()
plt.show()


## Question 8: Flatten one image into one vector

Take the grayscale crop and turn it into a one-dimensional vector.


In [ ]:
def flatten_image(image: np.ndarray) -> np.ndarray:
    # TODO: return a 1D vector version of the image.
    raise NotImplementedError("Flatten the image into one vector.")


sample_flat = flatten_image(sample_gray)
print("original shape:", sample_gray.shape)
print("flat shape:", sample_flat.shape)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(sample_flat[:256], color="#4C6FFF")
ax.set_title("First 256 grayscale values after flattening")
ax.set_xlabel("Index")
ax.set_ylabel("Value")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## Question 9: Engineer a feature vector with `concatenate` and `apply`

Build one hand-crafted feature vector that combines:

- RGB means
- RGB standard deviations
- the brightest channel index
- the mean and standard deviation of the filtered response
- one summary from `np.apply_along_axis(...)`

Use `np.concatenate(...)` to join the pieces.


In [ ]:
FEATURE_NAMES = [
    "mean_r",
    "mean_g",
    "mean_b",
    "std_r",
    "std_g",
    "std_b",
    "brightest_channel",
    "edge_mean",
    "edge_std",
    "row_std_mean",
]


def extract_features(image: np.ndarray) -> np.ndarray:
    cropped = center_crop(image, crop_size=64)
    image_float = normalize_01(cropped)
    gray = rgb_to_gray(image_float)
    channel_means, brightest_channel = channel_summary(image_float)
    channel_stds = image_float.std(axis=(0, 1)).astype(np.float32)
    filtered = convolve2d_matmul(gray, EDGE_KERNEL)
    row_std_profile = np.apply_along_axis(np.std, 1, gray)

    # TODO: use np.concatenate to combine the feature pieces into one float32 vector.
    raise NotImplementedError("Build one hand-crafted feature vector.")


sample_features = extract_features(sample_image)
print("feature shape:", sample_features.shape)
fig, ax = plot_feature_vector(sample_features, FEATURE_NAMES, title="Sample NumPy feature vector")
plt.show()


## Question 10: Build a tiny nearest-centroid classifier

Apply your feature function to a small balanced train/test subset from the original dataset.

Tasks:

1. build one feature matrix for the train images and one for the test images
2. compute one centroid for cats and one centroid for dogs
3. predict each test image by nearest centroid
4. measure accuracy
5. save the predictions to `artifacts/lab1_numpy_original_predictions_<student_id>.csv`


In [ ]:
def build_feature_matrix(paths: list[Path]) -> tuple[np.ndarray, np.ndarray]:
    # TODO: apply extract_features to every path and return X (features) and y (integer labels).
    raise NotImplementedError("Build the feature matrix for the nearest-centroid baseline.")


def nearest_centroid_predict(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    cat_centroid = X_train[y_train == 0].mean(axis=0)
    dog_centroid = X_train[y_train == 1].mean(axis=0)
    centroids = np.stack([cat_centroid, dog_centroid]).astype(np.float32)
    distances = np.linalg.norm(X_test[:, None, :] - centroids[None, :, :], axis=2)
    pred = np.argmin(distances, axis=1)
    return pred.astype(np.int64), centroids


X_train, y_train = build_feature_matrix(train_paths)
X_test, y_test = build_feature_matrix(test_paths)
test_pred, centroids = nearest_centroid_predict(X_train, y_train, X_test)
test_accuracy = float((y_test == test_pred).mean())
print(f"NumPy nearest-centroid accuracy: {test_accuracy:.3f}")

with NUMPY_PRED_PATH.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["filepath", "true_label", "pred_label", "correct"])
    writer.writeheader()
    for path, y_true, y_hat in zip(test_paths, y_test, test_pred):
        writer.writerow(
            {
                "filepath": str(path.relative_to(DATA_ROOT)),
                "true_label": LABELS[int(y_true)],
                "pred_label": LABELS[int(y_hat)],
                "correct": int(y_true == y_hat),
            }
        )

print(f"Saved predictions to: {NUMPY_PRED_PATH}")
fig, ax = plot_centroid_heatmap(centroids, FEATURE_NAMES, class_names=LABELS, title="Nearest-centroid feature means")
plt.show()
fig, axes = plot_prediction_gallery(
    test_paths,
    [LABELS[int(y)] for y in y_test],
    [LABELS[int(y)] for y in test_pred],
    load_image_np,
    max_items=8,
    ncols=4,
    figsize=(10, 6),
)
plt.show()


## Reflection

Answer these short questions in your own words:

1. Why is it useful to keep the crop size fixed before feature extraction?
2. What does `axis=(0, 1)` mean when you compute channel means on an image?
3. What information does the small edge filter capture that plain RGB means miss?
4. Why does flattening help some operations but also lose spatial structure?
